In [2]:
import sys
sys.path.append("../")

In [ ]:
import json
from pathlib import Path
from benchmark.eval import _ground_truth_dict
from spm import YOLOModel, ModelConfig

In [ ]:
# Initialize the YOLO model with the best weights
config = ModelConfig(
        model_path="../runs/segment/yolo-seg-whu/weights/best.pt",
        device="cuda",
        batch_size=4,
        confidence_threshold=0.25,
        tile_size=1500,
        overlap=0.1
        )

yolo = YOLOModel(config)

### Prepare data for hyperparameter tuning

In [ ]:
# Create pred_dir, gt_dir, and img_dir directories as expected by the spatial_mask_merging package
smm_optimize_dir = Path("smm_optimization_files")
smm_optimize_dir.mkdir(exist_ok=True)

pred_dir = smm_optimize_dir / "pred_dir"
pred_dir.mkdir(exist_ok=True)
gt_dir = smm_optimize_dir / "gt_dir"
gt_dir.mkdir(exist_ok=True)
img_dir = smm_optimize_dir / "img_dir"
img_dir.mkdir(exist_ok=True)

source_dir = Path("../benchmark_test_set")
crops = source_dir.glob("crop_3000_*")

In [ ]:
for crop in crops:
    image_path = crop / "image.tif"
    label_path = crop / "labels.gpkg"
    gt_path = gt_dir / f"{crop.name}.json"

    gt = _ground_truth_dict(label_path, image_path)
    gt["image_name"] = f"{crop.name}.tif"

    with open(gt_path, "w") as f:
        json.dump(gt, f)

    # Run inference on the image using the YOLO model
    pred = yolo.predict(image_path)
    pred_json = pred.to_json()
    pred_json["image_name"] = f"{crop.name}.tif"

    with open(pred_dir / f"{crop.name}.json", "w") as f:
        json.dump(pred_json, f)

    # Copy the image to the img_dir
    img_dest = img_dir / f"{crop.name}.tif"
    img_dest.write_bytes(image_path.read_bytes())

[2026-07-17 12:09:58.453176] INFO - Performing prediction on image: ../benchmark_test_set/crop_3000_3/image.tif (width: 3000, height: 3000) (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:79:predict())
[2026-07-17 12:10:00.271460] INFO - Total tiles processed: 9 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:169:predict())
[2026-07-17 12:10:00.515493] INFO - Performing prediction on image: ../benchmark_test_set/crop_3000_6/image.tif (width: 3000, height: 3000) (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:79:predict())
[2026-07-17 12:10:01.204813] INFO - Total tiles processed: 9 (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:169:predict())
[2026-07-17 12:10:01.310925] INFO - Performing prediction on image: ../benchmark_test_set/crop_3000_7/image.tif (width: 3000, height: 3000) (/media/vince/Backup/Projects/spatial_polygon_merging/src/spm/models/yolo.py:79:predict())
[

### Run the spatial mask merging optimization

In [ ]:
!cd ../spatial_mask_merging && \
PYTHONPATH=. uv run tools/optimize_smm.py \
--pred_dir ../notebooks/smm_optimization_files/pred_dir \
--gt_dir ../notebooks/smm_optimization_files/gt_dir \
--img_dir ../notebooks/smm_optimization_files/img_dir \
--out_dir ../notebooks/smm_optimization_files/opt_results \
--trials 20

Using CUDA for Evaluation: True (Device: cuda)
[I 2026-07-17 16:57:26,699] A new study created in memory with name: no-name-8dc4b5e7-bc8a-4fc3-a675-4d06f34cc1f5
Trial:   0%|                                             | 0/1 [00:00<?, ?it/s][ WARN:0@2.714] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@2.714] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@2.714] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@2.714] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@2.714] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Unknown field with tag 42113 (0xa481) encountered
[I 2026-07-17 16:58:11,280] Trial 0 finished with value: 0.8208955223377516 and parameters: {'tau_d': 14.363502971184062, 'tau_i': 0.8605714